# 01 — Define Plot Models / 01 Define Plot Models

**Chapter 18 — File 1 of 4 / 第18章 — 第1个文件（共4个）**

---

## Summary / 总结

This script demonstrates **create and plot the infogan model for mnist**.

本脚本演示 **create and plot the infogan model for mnist**。

---
## Step 1 — create and plot the infogan model for mnist

In [ ]:
from keras.optimizers import Adam
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import BatchNormalization
from keras.layers import Activation
from keras.initializers import RandomNormal
from keras.utils.vis_utils import plot_model

---
## Step 2 — define the standalone discriminator model

In [ ]:
def define_discriminator(n_cat, in_shape=(28,28,1)):

---
## Step 3 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 4 — image input

In [ ]:
in_image = Input(shape=in_shape)

---
## Step 5 — downsample to 14x14

In [ ]:
d = Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(in_image)
	d = LeakyReLU(alpha=0.1)(d)

---
## Step 6 — downsample to 7x7

In [ ]:
d = Conv2D(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = LeakyReLU(alpha=0.1)(d)
	d = BatchNormalization()(d)

---
## Step 7 — normal

In [ ]:
d = Conv2D(256, (4,4), padding='same', kernel_initializer=init)(d)
	d = LeakyReLU(alpha=0.1)(d)
	d = BatchNormalization()(d)

---
## Step 8 — flatten feature maps

In [ ]:
d = Flatten()(d)

---
## Step 9 — real/fake output

In [ ]:
out_classifier = Dense(1, activation='sigmoid')(d)

---
## Step 10 — define d model

In [ ]:
d_model = Model(in_image, out_classifier)

---
## Step 11 — compile d model

In [ ]:
d_model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.0002, beta_1=0.5))

---
## Step 12 — create q model layers

In [ ]:
q = Dense(128)(d)
	q = BatchNormalization()(q)
	q = LeakyReLU(alpha=0.1)(q)

---
## Step 13 — q model output

In [ ]:
out_codes = Dense(n_cat, activation='softmax')(q)

---
## Step 14 — define q model

In [ ]:
q_model = Model(in_image, out_codes)
	return d_model, q_model

---
## Step 15 — define the standalone generator model

In [ ]:
def define_generator(gen_input_size):

---
## Step 16 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 17 — image generator input

In [ ]:
in_lat = Input(shape=(gen_input_size,))

---
## Step 18 — foundation for 7x7 image

In [ ]:
n_nodes = 512 * 7 * 7
	gen = Dense(n_nodes, kernel_initializer=init)(in_lat)
	gen = Activation('relu')(gen)
	gen = BatchNormalization()(gen)
	gen = Reshape((7, 7, 512))(gen)

---
## Step 19 — normal

In [ ]:
gen = Conv2D(128, (4,4), padding='same', kernel_initializer=init)(gen)
	gen = Activation('relu')(gen)
	gen = BatchNormalization()(gen)

---
## Step 20 — upsample to 14x14

In [ ]:
gen = Conv2DTranspose(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(gen)
	gen = Activation('relu')(gen)
	gen = BatchNormalization()(gen)

---
## Step 21 — upsample to 28x28

In [ ]:
gen = Conv2DTranspose(1, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(gen)

---
## Step 22 — tanh output

In [ ]:
out_layer = Activation('tanh')(gen)

---
## Step 23 — define model

In [ ]:
model = Model(in_lat, out_layer)
	return model

---
## Step 24 — define the combined discriminator, generator and q network model

In [ ]:
def define_gan(g_model, d_model, q_model):

---
## Step 25 — make weights in the discriminator (some shared with the q model) as not trainable

In [ ]:
for layer in d_model.layers:
		if not isinstance(layer, BatchNormalization):
			layer.trainable = False

---
## Step 26 — connect g outputs to d inputs

In [ ]:
d_output = d_model(g_model.output)

---
## Step 27 — connect g outputs to q inputs

In [ ]:
q_output = q_model(g_model.output)

---
## Step 28 — define composite model

In [ ]:
model = Model(g_model.input, [d_output, q_output])

---
## Step 29 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss=['binary_crossentropy', 'categorical_crossentropy'], optimizer=opt)
	return model

---
## Step 30 — number of values for the categorical control code

In [ ]:
n_cat = 10

---
## Step 31 — size of the latent space

In [ ]:
latent_dim = 62

---
## Step 32 — create the discriminator

In [ ]:
d_model, q_model = define_discriminator(n_cat)

---
## Step 33 — create the generator

In [ ]:
gen_input_size = latent_dim + n_cat
g_model = define_generator(gen_input_size)

---
## Step 34 — create the gan

In [ ]:
gan_model = define_gan(g_model, d_model, q_model)

---
## Step 35 — plot the model

In [ ]:
plot_model(gan_model, to_file='gan_plot.png', show_shapes=True, show_layer_names=True)

---
## Learning Notes / 学习笔记

- **概念**: create and plot the infogan model for mnist 是机器学习中的常用技术。  
  *create and plot the infogan model for mnist is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Define Plot Models / 01 Define Plot Models
# Complete Code / 完整代码
# ===============================

# create and plot the infogan model for mnist
from keras.optimizers import Adam
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import BatchNormalization
from keras.layers import Activation
from keras.initializers import RandomNormal
from keras.utils.vis_utils import plot_model

# define the standalone discriminator model
def define_discriminator(n_cat, in_shape=(28,28,1)):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# image input
	in_image = Input(shape=in_shape)
	# downsample to 14x14
	d = Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(in_image)
	d = LeakyReLU(alpha=0.1)(d)
	# downsample to 7x7
	d = Conv2D(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = LeakyReLU(alpha=0.1)(d)
	d = BatchNormalization()(d)
	# normal
	d = Conv2D(256, (4,4), padding='same', kernel_initializer=init)(d)
	d = LeakyReLU(alpha=0.1)(d)
	d = BatchNormalization()(d)
	# flatten feature maps
	d = Flatten()(d)
	# real/fake output
	out_classifier = Dense(1, activation='sigmoid')(d)
	# define d model
	d_model = Model(in_image, out_classifier)
	# compile d model
	d_model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.0002, beta_1=0.5))
	# create q model layers
	q = Dense(128)(d)
	q = BatchNormalization()(q)
	q = LeakyReLU(alpha=0.1)(q)
	# q model output
	out_codes = Dense(n_cat, activation='softmax')(q)
	# define q model
	q_model = Model(in_image, out_codes)
	return d_model, q_model

# define the standalone generator model
def define_generator(gen_input_size):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# image generator input
	in_lat = Input(shape=(gen_input_size,))
	# foundation for 7x7 image
	n_nodes = 512 * 7 * 7
	gen = Dense(n_nodes, kernel_initializer=init)(in_lat)
	gen = Activation('relu')(gen)
	gen = BatchNormalization()(gen)
	gen = Reshape((7, 7, 512))(gen)
	# normal
	gen = Conv2D(128, (4,4), padding='same', kernel_initializer=init)(gen)
	gen = Activation('relu')(gen)
	gen = BatchNormalization()(gen)
	# upsample to 14x14
	gen = Conv2DTranspose(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(gen)
	gen = Activation('relu')(gen)
	gen = BatchNormalization()(gen)
	# upsample to 28x28
	gen = Conv2DTranspose(1, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(gen)
	# tanh output
	out_layer = Activation('tanh')(gen)
	# define model
	model = Model(in_lat, out_layer)
	return model

# define the combined discriminator, generator and q network model
def define_gan(g_model, d_model, q_model):
	# make weights in the discriminator (some shared with the q model) as not trainable
	for layer in d_model.layers:
		if not isinstance(layer, BatchNormalization):
			layer.trainable = False
	# connect g outputs to d inputs
	d_output = d_model(g_model.output)
	# connect g outputs to q inputs
	q_output = q_model(g_model.output)
	# define composite model
	model = Model(g_model.input, [d_output, q_output])
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss=['binary_crossentropy', 'categorical_crossentropy'], optimizer=opt)
	return model

# number of values for the categorical control code
n_cat = 10
# size of the latent space
latent_dim = 62
# create the discriminator
d_model, q_model = define_discriminator(n_cat)
# create the generator
gen_input_size = latent_dim + n_cat
g_model = define_generator(gen_input_size)
# create the gan
gan_model = define_gan(g_model, d_model, q_model)
# plot the model
plot_model(gan_model, to_file='gan_plot.png', show_shapes=True, show_layer_names=True)

---

➡️ **Next / 下一步**: File 2 of 4